# Start

In [1]:
%load_ext autoreload
%autoreload 2

## sensor_simulation.py

In [1]:
from datetime import datetime, timezone, timedelta
import random

def sensor_simulation() -> dict[str,float]:
  tz_wib = timezone(timedelta(hours=7))
  temperature = random.uniform(20.0, 40.0)
  air_humidity = random.uniform(40.0, 100.0)
  soil_moisture = random.uniform(0.0, 100.0)
  soil_ph = random.uniform(4.0, 7.0)

  return {
      'reading_timestamp': datetime.now(tz_wib).isoformat(),
      'temperature': temperature,
      'air_humidity': air_humidity,
      'soil_moisture': soil_moisture,
      'soil_ph': soil_ph
  }

## rsa_encryption.py

In [ ]:
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import hashes

def rsa_encrypt(message: bytes, public_key_pem: str) -> bytes:
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    ciphertext = public_key.encrypt(
        message,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return ciphertext

## get_public_key.py

In [3]:
import requests
from build_payload import build_payload

def get_public_key() -> str:
    url='http://localhost:3000/public-key'
    response = requests.get(url)
    response.raise_for_status()
    return response.json()['public_key']

c:\Users\theof\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
get_public_key()

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAt3XTsIWdBFs99kqlR41K\nqLOx3oZlo3F8UXsX1Z1EqF6P9AwgfKZ6ce9BeR/Qbvnt3RNSAO5az4Gj2JFSBGKF\n9yzVTVehLrVBlJVfouxO13vfwd8/gB4tpge8VqjGmvMJDN6sscPvJGxguEh/d7dV\nTGhJcnHKBlCqOVGqbrn6LzcXyuhRFiUisyELmpZByoe/EMwqKaLkl5tVWxZ2mYo9\nesjuteoJ9+3zAeYCtul6VIeDNx54SJjg7Ukmbh5UeRD4YO0fBxVuJOjJgKY/HI6k\nDCzzkNxHFs42rrX0V+W4FrjFspkJlTUEA/LfOVgZV+xuB1EBEISA79Zib/yOcKg1\nSwIDAQAB\n-----END PUBLIC KEY-----\n'

In [5]:
print(b'\xf8\xb1\xab)4\xac\xdf\xfa\x8b)\xbf\xd9\xd9^c\xc5\x17\x11\x8f\xfe{\xeb\x00u\xaa9\xac:\x8aVp\xcd')

b'\xf8\xb1\xab)4\xac\xdf\xfa\x8b)\xbf\xd9\xd9^c\xc5\x17\x11\x8f\xfe{\xeb\x00u\xaa9\xac:\x8aVp\xcd'


## build_payload.py

In [6]:
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from rsa_encryption import rsa_encrypt
from sensor_simulation import sensor_simulation

import base64
from random import randbytes

def generate_session_id(num_bytes = 32):
    return randbytes(num_bytes)

def build_payload(public_key_pem: str, send_no_session_key: bool = False, return_aad_bytes: bool = False) -> dict:
	tz_wib = timezone(timedelta(hours=7))
	session_key = generate_session_id()
	encrypted_session_key = rsa_encrypt(session_key, public_key_pem) # -> raw_bytes

	aesgcm = AESGCM(session_key)
	iv = os.urandom(12)
	plaintext_bytes = json.dumps(sensor_simulation()).encode('utf-8') # -> dict[str,float]
	aad = {
		'sensor_id': 1,
		'transmission_timestamp': datetime.now(tz_wib).isoformat(),
		'encrypted_session_key': base64.b64encode(encrypted_session_key).decode('utf-8') if not send_no_session_key else None,
	}
	aad_bytes = json.dumps(aad, separators=(',', ':'), sort_keys=True).encode('utf-8')

	encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
	ciphertext = encrypted_raw[:-16]
	tag = encrypted_raw[-16:]

	return {
		'aad': aad if not return_aad_bytes else aad_bytes.decode('utf-8'),
		'nonce': base64.b64encode(iv).decode('utf-8'),
		'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
		'tag': base64.b64encode(tag).decode('utf-8')
	}

## build_payload2.py (send raw aad)

In [7]:
# %%writefile build_payload2.py
# from datetime import datetime, timezone, timedelta
# import os
# from cryptography.hazmat.primitives.ciphers.aead import AESGCM
# import base64
# import json
# from rsa_encryption import rsa_encrypt
# from sensor_simulation import sensor_simulation


# def build_payload2(public_key_pem: str) -> dict:
# 	tz_wib = timezone(timedelta(hours=7))
# 	session_key = b'\xf8\xb1\xab)4\xac\xdf\xfa\x8b)\xbf\xd9\xd9^c\xc5\x17\x11\x8f\xfe{\xeb\x00u\xaa9\xac:\x8aVp\xcd'
# 	encrypted_session_key = rsa_encrypt(session_key, public_key_pem) # -> raw_bytes

# 	aesgcm = AESGCM(session_key)
# 	iv = os.urandom(12)
# 	plaintext_bytes = json.dumps(sensor_simulation()).encode('utf-8') # -> dict[str,float]
# 	aad = {
# 		'sensor_id': 1,
# 		'transmission_timestamp': datetime.now(tz_wib).isoformat(),
# 		'encrypted_session_key': base64.b64encode(encrypted_session_key).decode('utf-8'),
# 	}
# 	aad_bytes = json.dumps(aad).encode('utf-8')

# 	encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
# 	ciphertext = encrypted_raw[:-16]
# 	tag = encrypted_raw[-16:]

# 	return {
# 		'aad': aad_bytes.decode('utf-8'),
# 		'nonce': base64.b64encode(iv).decode('utf-8'),
# 		'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
# 		'tag': base64.b64encode(tag).decode('utf-8')
# 	}

## Send payload

In [8]:
import requests
import time
# from build_payload import build_payload
# from build_payload2 import build_payload2
# from get_public_key import get_public_key
import numpy as np
from scipy import stats
# from sensor_simulation import sensor_simulation
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

sensor_simulation()
server_public_key = get_public_key()

url = 'http://localhost:3000/telemetry'

# CPU warm up
for _ in range(10):
    build_payload(server_public_key)
    
durations1 = []

for _ in range(1):
    start = time.perf_counter()
    payload = build_payload(server_public_key)
    
    response=requests.post(url, json=payload)
    
    end = time.perf_counter()
    durations1.append(end - start)

print(response.json())

{'decryption': 'Success'}


# Stop

In [19]:
import json

aad = {
      'sensor_id': 1,
      'transmission_timestamp': 'datetime.now(tz_wib).isoformat()',
      'encrypted_session_key': 'session_key',
  }

aad_json = json.dumps(aad)
aad_bytes = aad_json.encode('utf-8')
print(aad_json)
print(aad_bytes)
print(type(aad_json))
print(type(aad_bytes))
# print(aad.encode('utf-8'))

{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}
b'{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}'
<class 'str'>
<class 'bytes'>


In [20]:
import base64
import json
random_bytes = b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'
print({'random_bytes': random_bytes})
print(type(random_bytes))
A = base64.b64encode(random_bytes)
B = A.decode('utf-8')
print({'random_bytes': A})
print(type(A))
print({'random_bytes': B})
print(type(B))
myjson= json.dumps({'random_bytes' : 1})
print(myjson)
print(type(myjson))

{'random_bytes': b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'}
<class 'bytes'>
{'random_bytes': b'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'bytes'>
{'random_bytes': 'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'str'>
{"random_bytes": 1}
<class 'str'>


In [21]:
mydict = {'random_bytes': B}
print(type(mydict))

<class 'dict'>
